# 00 - Setup and universe build

Run this once, top to bottom. It does everything before certificate collection:

1. mounts Drive, clones the code, creates the Drive folder tree
2. finds the raw downloads (filenames are auto-detected - nothing to edit)
3. loads every source into one labelled corpus
4. reports family coverage and class balance
5. writes `domains_labelled.parquet` and `probe_universe.parquet`

**Nothing needs editing.** If a source is missing it is skipped with a note.

Storage note: Pro+ gives 2 TB of *Drive*, not RAM. Training data is staged to
`/content` and read from local SSD.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'

if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{URL}', REPO], check=True)

sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

**If you edited code on GitHub while this session was running:** the pull
above updates the file on disk, but Python keeps the old module in memory.
Do Runtime -> Restart session, then run from the top. A stale module is the
single most confusing failure mode here - it produces wrong output with no
error.

In [ ]:
!pip -q install dnspython cryptography xgboost shap pyarrow zstandard pyyaml

In [ ]:
from pathlib import Path
RAW = Path(P['data']['raw'])
INTERIM = Path(P['data']['interim'])

for f in sorted(RAW.iterdir()):
    size = '   DIR' if f.is_dir() else f'{f.stat().st_size/1e6:6.1f}MB'
    print(size, f.name)

## Locate the sources

Filenames are discovered rather than hard-coded. When several dated snapshots
of a live feed exist, the most recent is used and the others are ignored -
mixing snapshots would make the capture date unreportable.

In [ ]:
def newest(pattern, root=RAW):
    hits = sorted(root.glob(pattern))
    return hits[-1] if hits else None

TRANCO    = newest('tranco_*.csv')
OPENPHISH = newest('openphish_*.txt')
URLHAUS   = newest('urlhaus_*.csv')
UMUDGA    = RAW/'umudga'/'Fully Qualified Domain Names'
DGARCHIVE = newest('dgarchive*.csv')          # None until FKIE approves
CICBELL   = newest('cic_bell*.csv')           # None until UNB approves

for name, p in [('tranco', TRANCO), ('umudga', UMUDGA), ('openphish', OPENPHISH),
                ('urlhaus', URLHAUS), ('dgarchive', DGARCHIVE), ('cic_bell', CICBELL)]:
    ok = p is not None and Path(p).exists()
    print(f'{name:10s} {"FOUND  " if ok else "missing"} {p if ok else ""}')

## Load

UMUDGA ships a family called `legit` - their benign control set, not a DGA.
It is excluded here: labelling it malicious would inject roughly 20k benign
domains into the positive class and corrupt the ground truth.

In [ ]:
from src.data import universe as U

frames = []

# --- benign ---
if TRANCO:
    frames.append(U.load_tranco(TRANCO))          # full 1M depth

# --- malicious ---
if UMUDGA.exists():
    dga_families = sorted({d.parent.name.lower() for d in UMUDGA.rglob('list')
                           if d.parent.name.lower() != 'legit'})
    print('DGA families (excluding UMUDGA "legit" control set):', len(dga_families))
    frames.append(U.load_umudga(UMUDGA, families=dga_families))

if OPENPHISH:
    frames.append(U.load_phish_feed(str(OPENPHISH), 'openphish', 'phishing'))
if URLHAUS:
    frames.append(U.load_phish_feed(str(URLHAUS), 'urlhaus', 'malware'))
if DGARCHIVE:
    frames.append(U.load_dgarchive(DGARCHIVE))
if CICBELL:
    frames.append(U.load_cic_bell(CICBELL))

print()
for f in frames:
    print(f'{f["source"].iloc[0]:22s} {len(f):>9,}  families: {f["family"].nunique()}')

## Combine

A domain appearing as both benign and malicious is a label conflict. These are
dropped rather than arbitrated - keeping them injects noise into the ground
truth that no model can overcome, and it would show up as irreducible error.

In [ ]:
df = U.combine(frames)

print(f'total rows        {len(df):>10,}')
print(f'benign            {(df.label==0).sum():>10,}')
print(f'malicious         {(df.label==1).sum():>10,}')
print(f'imbalance         1 malicious : {(df.label==0).sum()/max((df.label==1).sum(),1):.1f} benign')
print(f'conflicts dropped {df.attrs.get("n_conflicts_dropped", 0):>10,}')
print()
display(df.groupby('source')['label'].agg(n='count', malicious_rate='mean').round(3))

### Family coverage

The family-disjoint split holds out whole DGA families to measure
generalisation to families never seen in training. It needs enough families to
be stable - roughly ten or more with a few hundred domains each. Fewer than
that, and the split should be reported as a limitation rather than quietly
replaced by random splitting.

In [ ]:
fam = df[df.label==1]['family'].value_counts()
print('distinct families:      ', len(fam))
print('families with >=500:    ', (fam>=500).sum())
print('smallest / largest:     ', fam.min(), '/', fam.max())
display(fam.head(20))

In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)
out = INTERIM/'domains_labelled.parquet'
df.to_parquet(out, index=False, compression='zstd')
print('wrote', out, df.shape)

## Probe universe

The certificate probe is the expensive, time-bounded step, so it runs on a
stratified subset. Malicious rows are sampled per family so no single large
family dominates and the family-disjoint split stays viable.

50k domains at ~100 concurrent probes is roughly 2-4 hours of wall clock.

Expect most DGA domains to return NXDOMAIN: they were generated but never
registered. That is not a failure - it is why the lexical branch carries DGA
detection while the certificate branch carries live malicious infrastructure
from URLhaus and OpenPhish. The fusion covers both regimes.

In [ ]:
probe = U.probe_universe(df, n_total=50_000, malicious_fraction=0.30, seed=42)
probe.to_parquet(INTERIM/'probe_universe.parquet', index=False, compression='zstd')

print(f'probe set         {len(probe):>10,}')
print(f'  benign          {(probe.label==0).sum():>10,}')
print(f'  malicious       {(probe.label==1).sum():>10,}')
print(f'  families        {probe[probe.label==1]["family"].nunique():>10}')
print()
display(probe[probe.label==1].groupby('source').size())

## Record the snapshot identity

These strings go into the paper's data section. They are recoverable today and
not recoverable later.

In [ ]:
import datetime, json
prov = {
    'built_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'tranco_file': TRANCO.name if TRANCO else None,
    'tranco_list_url': f'https://tranco-list.eu/list/{TRANCO.stem.split("_")[-1]}' if TRANCO else None,
    'openphish_snapshot': OPENPHISH.name if OPENPHISH else None,
    'urlhaus_snapshot': URLHAUS.name if URLHAUS else None,
    'umudga_families': len(dga_families) if UMUDGA.exists() else 0,
    'umudga_note': 'UMUDGA "legit" control family excluded from malicious class',
    'n_total': int(len(df)), 'n_benign': int((df.label==0).sum()),
    'n_malicious': int((df.label==1).sum()),
    'n_label_conflicts_dropped': int(df.attrs.get('n_conflicts_dropped', 0)),
    'probe_n': int(len(probe)),
}
(INTERIM/'universe_provenance.json').write_text(json.dumps(prov, indent=2))
print(json.dumps(prov, indent=2))

---

**Next:** open `02_certificate_collection.ipynb`, start it, and leave it
running. While it runs, work through `01_data_audit` (leakage screen),
`04_split_creation`, and baselines on lexical features - none of those need
certificate data.